# 03_eda — Exploratory Data Analysis

Explore the processed feature table (`data/processed/`) to understand
**what relates to the listing multiple**.

**What we look at**
1. Target distribution (raw vs log) — does the log transform tame the right skew?
2. Multiple by monetization / niche / country
3. Business age & profit scale vs multiple
4. Feature correlations (multicollinearity check)

> This is exploration. No cleaning or modeling here — just looking.


In [ ]:
import ast
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parents[1] if (Path.cwd().name == "local") else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Load the most recent processed feature table
files = sorted(PROCESSED_DIR.glob("features_*.csv"))
if not files:
    raise FileNotFoundError("No processed features found. Run 02_clean_features first.")
df = pd.read_csv(files[-1])
print("Loaded:", files[-1].name)
print("shape:", df.shape)

TARGET = "annual_listing_multiple"


## 1. Target Distribution — Raw vs Log

In 01 we saw the target is right-skewed (long tail toward high multiples).
Compare the raw target with the log-transformed target side by side.
A more symmetric log distribution justifies modeling on `log_target`.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].hist(df[TARGET].dropna(), bins=50, color="#4C72B0", edgecolor="white")
ax[0].set_title("Raw multiple")
ax[0].set_xlabel("annual_listing_multiple")
ax[0].set_ylabel("count")

ax[1].hist(df["log_target"].dropna(), bins=50, color="#55A868", edgecolor="white")
ax[1].set_title("Log multiple  (log1p)")
ax[1].set_xlabel("log_target")

plt.tight_layout()
plt.show()

print("Raw  :", df[TARGET].describe().round(2).to_dict())
print("Log  :", df["log_target"].describe().round(2).to_dict())


## 2. Multiple by Monetization Type

`monetizations` is a list per listing. Here we take the **primary** (first) type
just for grouping, and compare the average multiple across types.
This hints at which business models command higher multiples.


In [ ]:
def first_monetization(x):
    try:
        lst = ast.literal_eval(x) if isinstance(x, str) else x
        if isinstance(lst, list) and lst:
            item = lst[0]
            return item.get("monetization") if isinstance(item, dict) else item
    except Exception:
        pass
    return "Unknown"

df["primary_monetization"] = df["monetizations"].apply(first_monetization)

counts = df["primary_monetization"].value_counts()
means = df.groupby("primary_monetization")[TARGET].mean().sort_values(ascending=False)

# Only show types with enough samples (>= 5) to be meaningful
common = counts[counts >= 5].index
means_common = means[means.index.isin(common)]

fig, ax = plt.subplots(figsize=(9, 4))
means_common.plot(kind="bar", ax=ax, color="#4C72B0", edgecolor="white")
ax.set_title("Average multiple by monetization (types with >= 5 listings)")
ax.set_ylabel("avg annual_listing_multiple")
ax.set_xlabel("")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

print("Counts:")
print(counts.head(10).to_string())


## 3. Multiple by Niche

Same idea for niches. Niches can have high cardinality, so we show the
top niches by listing count and their average multiple.


In [ ]:
def first_niche(x):
    try:
        lst = ast.literal_eval(x) if isinstance(x, str) else x
        if isinstance(lst, list) and lst:
            item = lst[0]
            return item.get("niche") if isinstance(item, dict) else item
    except Exception:
        pass
    return "Unknown"

df["primary_niche"] = df["niches"].apply(first_niche)

niche_counts = df["primary_niche"].value_counts()
top_niches = niche_counts[niche_counts >= 10].index   # niches with >= 10 listings
niche_means = df[df["primary_niche"].isin(top_niches)].groupby("primary_niche")[TARGET].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
niche_means.plot(kind="barh", ax=ax, color="#55A868", edgecolor="white")
ax.set_title("Average multiple by niche (niches with >= 10 listings)")
ax.set_xlabel("avg annual_listing_multiple")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## 4. Business Age & Profit Scale vs Multiple

Scatter plots to see whether older or larger businesses tend to sell at higher multiples.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].scatter(df["business_age_months"], df[TARGET], s=8, alpha=0.3, color="#4C72B0")
ax[0].set_title("Business age vs multiple")
ax[0].set_xlabel("business_age_months")
ax[0].set_ylabel("annual_listing_multiple")

ax[1].scatter(df["log_average_annual_net_profit"], df[TARGET], s=8, alpha=0.3, color="#C44E52")
ax[1].set_title("Net profit (log) vs multiple")
ax[1].set_xlabel("log_average_annual_net_profit")

plt.tight_layout()
plt.show()


## 5. Feature Correlations (Multicollinearity Check)

Correlation of key numeric features with each other and with `log_target`.
- Look at the `log_target` row/column to see what moves with the target.
- High correlation BETWEEN two features signals redundancy (multicollinearity).


In [ ]:
num_feats = [
    "log_target",
    "log_average_annual_net_profit",
    "log_average_annual_gross_revenue",
    "business_age_months",
    "net_margin",
    "expense_ratio",
    "hours_worked_per_week",
    "days_on_marketplace",
    "monetizations_count",
    "niches_count",
    "profit_margin",
]
num_feats = [c for c in num_feats if c in df.columns]

corr = df[num_feats].corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(num_feats)))
ax.set_yticks(range(len(num_feats)))
ax.set_xticklabels(num_feats, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(num_feats, fontsize=8)
# annotate
for i in range(len(num_feats)):
    for j in range(len(num_feats)):
        ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if abs(corr.iloc[i,j]) > 0.5 else "black")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title("Feature correlation matrix")
plt.tight_layout()
plt.show()

print("Correlation with log_target (sorted):")
print(corr["log_target"].sort_values(ascending=False).round(3).to_string())


## 6. Text Length vs Multiple (optional)

Does a longer opportunities/risks description relate to the multiple?
Requires the PRIVATE text file (data/private/). If it's not present, this cell is skipped.
Text length here is a cheap proxy; real text features come in step 04 (embeddings).


In [ ]:
PRIVATE_DIR = PROJECT_ROOT / "data" / "private"
text_files = sorted(PRIVATE_DIR.glob("text_private_*.csv")) if PRIVATE_DIR.exists() else []

if text_files:
    tdf = pd.read_csv(text_files[-1])
    # join on listing id
    merged = df.merge(tdf, on="id", how="left", suffixes=("", "_txt"))
    for col in ["opportunities", "risks", "summary"]:
        if col in merged.columns:
            merged[f"{col}_len"] = merged[col].astype(str).str.len()

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.scatter(merged["summary_len"], merged[TARGET], s=8, alpha=0.3, color="#8172B3")
    ax.set_title("Summary length vs multiple")
    ax.set_xlabel("summary length (chars)")
    ax.set_ylabel("annual_listing_multiple")
    plt.tight_layout()
    plt.show()

    print("Correlation (summary_len vs multiple):",
          round(merged["summary_len"].corr(merged[TARGET]), 3))
else:
    print("No private text file found — skipping text-length analysis.")


## 7. Takeaways

Note down what stood out (fill in after running):
- Does the log transform look justified?
- Which monetizations / niches show higher multiples?
- Which features correlate most with `log_target`?
- Any redundant (highly correlated) feature pairs to watch in modeling?

**Next: `05_model_baseline`** — a tabular-only baseline
(regularized linear + gradient boosting), cross-validated,
reporting MAE/RMSE on the log target plus error in original units.
